# my-coding-agent — Sample Notebook

Demonstrates the  library: tool registration, single-shot LLM calls, and the full agentic loop.

**Prerequisites:** local OMLX server running at  and the package installed:


## 1. Imports

In [14]:
import importlib
import my_coding_agent

importlib.reload(my_coding_agent)

from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

## 2. Inspect a tool definition

The  converter turns any typed Python function into an OpenAI-compatible schema.

In [2]:
import json

# Built-in example tool
schema = tool(ToolsRegistry.get_weather)
print(json.dumps(schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the current weather for a location.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string"
        }
      },
      "required": [
        "location"
      ]
    }
  }
}


## 3. Register a custom tool

Define a new function and register it on  at runtime.

In [3]:
def get_stock_price(ticker: str, currency: str = "USD") -> str:
    """Get the current stock price for a ticker symbol."""
    # stub — replace with a real API call
    prices = {"AAPL": 213.5, "GOOGL": 175.2, "MSFT": 425.0}
    price = prices.get(ticker.upper(), 0.0)
    return f"{ticker.upper()} is trading at {price} {currency}"

# Attach to registry so the LLM can call it
ToolsRegistry.get_stock_price = staticmethod(get_stock_price)

print(json.dumps(tool(get_stock_price), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_stock_price",
    "description": "Get the current stock price for a ticker symbol.",
    "parameters": {
      "type": "object",
      "properties": {
        "ticker": {
          "type": "string"
        },
        "currency": {
          "type": "string"
        }
      },
      "required": [
        "ticker"
      ]
    }
  }
}


## 4. Single-shot LLM call (no tools)

Use  directly for a plain chat completion.

In [4]:
llm = LLM(api_url=OMLX_API_URL, api_key=OMLX_API_KEY, model=OMLX_MODEL)

messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user",   "content": "What is 42 in binary?"}
]

resp = llm.chat_completion(messages)
msg = extract_message(resp)
print(msg["content"])

2026-05-22 22:03:47 | INFO     | my_coding_agent.llm | Available models: ['MLX-Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'MLX-Qwen3.5-9B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit', 'Qwen3.5-9B-MLX-4bit', 'Qwen3.6-27B-4bit', 'Qwen3.6-35B-A3B-4bit', 'Qwen3.6-35B-A3B-5bit', 'Qwen3.6-35B-A3B-6bit', 'gemma-4-26b-a4b-it-4bit', 'gemma-4-31b-it-4bit', 'gemma-4-e2b-it-4bit', 'gpt-oss-20b-MXFP4-Q4', 'gpt-oss-20b-MXFP4-Q8']
2026-05-22 22:03:47 | INFO     | my_coding_agent.llm | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 22:03:51 | INFO     | my_coding_agent.llm | Received response: 200 (1516 bytes)
2026-05-22 22:03:51 | DEBUG    | my_coding_agent.llm | Response content: {'id': 'chatcmpl-5fd04791', 'object': 'chat.completion', 'created': 1779480231, 'model': 'Qwen3.6-35B-A3B-4bit', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '42 in binary is **10101

## 5. Single-shot call with tool use

Pass tools manually and inspect the tool-call response, then execute it.

In [5]:
tools = [
    tool(ToolsRegistry.get_weather),
    tool(ToolsRegistry.get_stock_price),
]
print("Tools schemas: ", json.dumps(tools, indent=2))

messages = [
    {"role": "system", "content": "Use the available tools to answer."},
    {"role": "user",   "content": "What is the weather in Tokyo? And what is the price of AAPL stock?"}
]

resp = llm.chat_completion(messages, tools=tools)
msg = extract_message(resp)
print("finish_reason:", extract_finish_reason(resp))
print("tool_calls:", json.dumps(msg.get("tool_calls"), indent=2))

Tools schemas:  [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get the current weather for a location.",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string"
          }
        },
        "required": [
          "location"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_stock_price",
      "description": "Get the current stock price for a ticker symbol.",
      "parameters": {
        "type": "object",
        "properties": {
          "ticker": {
            "type": "string"
          },
          "currency": {
            "type": "string"
          }
        },
        "required": [
          "ticker"
        ]
      }
    }
  }
]
2026-05-22 22:03:51 | INFO     | my_coding_agent.llm | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 22:03:58 | INFO     | my_coding_

In [6]:
# Execute the tool calls and get the final answer
messages.append(msg)
tool_results = llm.execute_tool_calls(msg)
messages.extend(tool_results)

final_resp = llm.chat_completion(messages)
print(extract_message(final_resp)["content"])

2026-05-22 22:03:58 | INFO     | my_coding_agent.llm | Processing tool call: get_weather with args {'location': 'Tokyo'}
2026-05-22 22:03:58 | INFO     | my_coding_agent.llm | Executed tool: get_weather with args {'location': 'Tokyo'}
2026-05-22 22:03:58 | INFO     | my_coding_agent.llm | Processing tool call: get_stock_price with args {'ticker': 'AAPL'}
2026-05-22 22:03:58 | INFO     | my_coding_agent.llm | Executed tool: get_stock_price with args {'ticker': 'AAPL'}
2026-05-22 22:03:58 | INFO     | my_coding_agent.llm | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 22:03:59 | INFO     | my_coding_agent.llm | Received response: 200 (941 bytes)
2026-05-22 22:03:59 | DEBUG    | my_coding_agent.llm | Response content: {'id': 'chatcmpl-150efff0', 'object': 'chat.completion', 'created': 1779480239, 'model': 'Qwen3.6-35B-A3B-4bit', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'The current weather in Tokyo is sunny

## 6. Full agentic loop with 

 handles the chat → tool-execute → respond cycle automatically.

In [ ]:
from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Use tools when needed. "
            "Available tools: get_weather(location), get_stock_price(ticker, currency)."
        )
    },
    {
        "role": "user", 
        "content": "What is the weather in London and the price of AAPL stock?"
    }
]

agent = Agent(
    api_url=OMLX_API_URL,
    api_key=OMLX_API_KEY,
    model=OMLX_MODEL,
    messages=messages,
    tools=tools,
)

final_messages = agent.run(max_steps=5)
print(" Final messages: ", json.dumps(final_messages, indent=2))

2026-05-22 22:03:59 | INFO     | my_coding_agent.llm | Available models: ['MLX-Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'MLX-Qwen3.5-9B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit', 'Qwen3.5-9B-MLX-4bit', 'Qwen3.6-27B-4bit', 'Qwen3.6-35B-A3B-4bit', 'Qwen3.6-35B-A3B-5bit', 'Qwen3.6-35B-A3B-6bit', 'gemma-4-26b-a4b-it-4bit', 'gemma-4-31b-it-4bit', 'gemma-4-e2b-it-4bit', 'gpt-oss-20b-MXFP4-Q4', 'gpt-oss-20b-MXFP4-Q8']
2026-05-22 22:03:59 | INFO     | my_coding_agent.agent | Agent initialized with API URL: http://127.0.0.1:8321/v1, Model: Qwen3.6-35B-A3B-4bit
2026-05-22 22:03:59 | INFO     | my_coding_agent.agent | Agent run started with max_steps: 5
2026-05-22 22:03:59 | INFO     | my_coding_agent.agent | ---------------- Step 1/5
2026-05-22 22:03:59 | INFO     | my_coding_agent.agent | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 22:04:02 | INFO     | my_coding_agent.

# 7. Agentic Bash Execution

In [8]:
import subprocess
import os
from my_coding_agent._logging import get_logger
from my_coding_agent.tools import ToolsRegistry, tool


def bash(command: str) -> str:
    """
    Execute a bash command and return its output.
    When working with paths, use absolute paths to avoid issues with the current working directory.
    """
    logger = get_logger("bash")
    logger.debug("Executing bash command: %s", command)
    cwd = os.getcwd()
    
    try:
        result = subprocess.run(
            command,
            cwd=cwd,
            shell=True,
            text=True,
            env=os.environ,
            encoding="utf-8",
            errors="replace",
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            timeout=30,
        )
        output = {"output": result.stdout, "returncode": result.returncode, "exception_info": ""}
    except Exception as e:
        returncode = getattr(e, "returncode", -1)
        output = getattr(e, "output", "")
        output = {"output": output, "returncode": returncode, "exception_info": "An error occurred while executing the bash command: " + str(e)}
    
    return output

# Attach to registry so the LLM can call it
ToolsRegistry.bash = staticmethod(bash)
print(json.dumps(tool(bash), indent=2))

# example
print(json.dumps(bash("ls -al"), indent=2))
print(json.dumps(bash("echo Hello World"), indent=2))

{
  "type": "function",
  "function": {
    "name": "bash",
    "description": "\n    Execute a bash command and return its output.\n    When working with paths, use absolute paths to avoid issues with the current working directory.\n    ",
    "parameters": {
      "type": "object",
      "properties": {
        "command": {
          "type": "string"
        }
      },
      "required": [
        "command"
      ]
    }
  }
}
2026-05-22 22:04:04 | DEBUG    | bash | Executing bash command: ls -al
{
  "output": "total 80\ndrwxr-xr-x@ 13 noordeepsingh  staff    416 May 22 17:11 \u001b.\u001b[m\u001b[m\ndrwxr-xr-x  58 noordeepsingh  staff   1856 May 21 23:10 \u001b..\u001b[m\u001b[m\ndrwxr-xr-x@  3 noordeepsingh  staff     96 May 22 16:56 \u001b.archive\u001b[m\u001b[m\ndrwxr-xr-x@  3 noordeepsingh  staff     96 May 22 01:51 \u001b.claude\u001b[m\u001b[m\ndrwxr-xr-x@ 14 noordeepsingh  staff    448 May 22 22:04 \u001b.git\u001b[m\u001b[m\n-rw-r--r--@  1 noordeepsingh  staff    159 May 22 

In [9]:
dir(ToolsRegistry)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'bash',
 'get_stock_price',
 'get_weather']

In [ ]:
from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Use tools when needed. "
            "Available tools: bash(command)"
        )
    },
    {
        "role": "user", 
        "content": "Please git commit and push the latest changes to my GitHub repository using the gh CLI tool. Then, list the files in the current directory to confirm the changes were pushed. with git status and git log."
    }
]
tools = [
    tool(ToolsRegistry.bash),
]

agent = Agent(
    api_url=OMLX_API_URL,
    api_key=OMLX_API_KEY,
    model=OMLX_MODEL,
    messages=messages,
    tools=tools,
)

final_messages = agent.run(max_steps=5)
print("Final messages: ", json.dumps(final_messages, indent=2))

2026-05-22 22:27:33 | INFO       | Agent      | Available models: ['MLX-Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'MLX-Qwen3.5-9B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit', 'Qwen3.5-9B-MLX-4bit', 'Qwen3.6-27B-4bit', 'Qwen3.6-35B-A3B-4bit', 'Qwen3.6-35B-A3B-5bit', 'Qwen3.6-35B-A3B-6bit', 'gemma-4-26b-a4b-it-4bit', 'gemma-4-31b-it-4bit', 'gemma-4-e2b-it-4bit', 'gpt-oss-20b-MXFP4-Q4', 'gpt-oss-20b-MXFP4-Q8']
2026-05-22 22:27:33 | INFO       | Agent      | Agent initialized with API URL: http://127.0.0.1:8321/v1, Model: Qwen3.6-35B-A3B-4bit
2026-05-22 22:27:33 | INFO       | Agent      | Agent run started with max_steps: 5
2026-05-22 22:27:33 | INFO       | Agent      | ----------------------------------------------------------------
2026-05-22 22:27:33 | INFO       | Agent      | ----------------------------------------------------------------   STEP 1/5
2026-05-22 22:27:33 | INFO       | Agent      | -------------------